In [1]:
import yaml
from pathlib import Path
import geopandas as gpd
import pandas as pd
from shapely.geometry import box
import numpy as np

In [3]:
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Gates Foundation/Building Dataset Validation"
)

CONFIG_PATH = PROJECT_ROOT / "configs/ssd-juba.yaml"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("CONFIG_PATH =", CONFIG_PATH)

with open(CONFIG_PATH, "r") as f:
    cfg = yaml.safe_load(f)

ROOT = (PROJECT_ROOT / cfg["root_dir"]).resolve()
CITY = cfg["city"]

CRS = cfg["crs"]
AOI_PATH = ROOT / cfg["aoi_path"]
REF_PATH = ROOT / cfg["reference_vector"]
CANDIDATES = cfg["candidates"]

MIN_AREA_M2 = cfg.get("min_area_m2", 10)
TILE_SIZE_M = cfg.get("tile_size_m", 1000)  # Default to 1km tiles

# IoU threshold
TAU_OVERLAP = cfg.get("tau_overlap", 0.5)

# Misalignment tolerance (buffer) for vector matching.
# In juba.yaml this is called tau_buffer_m.
TAU_BUFFER_M = cfg.get("tau_buffer_m", cfg.get("tau_misalign_m", 0.0))

# Boundary match threshold (meters). Accept both keys for backward compatibility.
TAU_BOUNDARY_M = cfg.get("tau_boundary_m", cfg.get("tau_boundary", 2.0))

print("City:", CITY)
print("Root:", ROOT)
print("CRS:", CRS)
print("MIN_AREA_M2:", MIN_AREA_M2)
print("TILE_SIZE_M:", TILE_SIZE_M)
print("TAU_OVERLAP:", TAU_OVERLAP)
print("TAU_BUFFER_M:", TAU_BUFFER_M)
print("TAU_BOUNDARY_M:", TAU_BOUNDARY_M)


PROJECT_ROOT = /content/drive/MyDrive/Gates Foundation/Building Dataset Validation
CONFIG_PATH = /content/drive/MyDrive/Gates Foundation/Building Dataset Validation/configs/ssd-juba.yaml
City: ssd-juba
Root: /content/drive/.shortcut-targets-by-id/1xTSKkjN4wft_0kljbwaAGKjTGEBsHZm1/Gates Foundation/Building Dataset Validation
CRS: EPSG:32636
MIN_AREA_M2: 20
TILE_SIZE_M: 1000
TAU_OVERLAP: 0.5
TAU_BUFFER_M: 2.0
TAU_BOUNDARY_M: 2.0


In [4]:
FIX_INVALID_GEOMS = True

In [ ]:
aoi = load_aoi(AOI_PATH, CRS)
tiles = make_tiles(aoi, TILE_SIZE_M)

# Save to 02_interim so we can reuse the tiling later
tiles_path = ROOT / f"data/02_interim/tiles/{CITY.lower()}_tiles.gpkg"
tiles_path.parent.mkdir(parents=True, exist_ok=True)
tiles.to_file(tiles_path, driver="GPKG")

tiles.head()


In [ ]:
# Load reference and make spatial index once
ref_all = load_buildings(REF_PATH, CRS, MIN_AREA_M2)
ref_all_sindex = ref_all.sindex

print("Reference buildings:", len(ref_all))

In [ ]:
# Ensure outputs folder
metrics_dir = ROOT / f"outputs/metrics/{CITY.lower()}"
metrics_dir.mkdir(parents=True, exist_ok=True)
figures_dir = ROOT / f"outputs/figures/{CITY.lower()}"
figures_dir.mkdir(parents=True, exist_ok=True)

all_tile_metrics = []
all_match_rows = []

for cand_cfg in CANDIDATES:
    ds_name = cand_cfg["name"]
    cand_path = ROOT / cand_cfg["path"]

    print(f"\n=== Dataset: {ds_name} ===")
    cand_all = load_buildings(cand_path, CRS, MIN_AREA_M2)
    cand_all_sindex = cand_all.sindex
    print("Candidate buildings:", len(cand_all))

    ds_tile_metrics = []
    ds_match_rows = []

    for _, tile_row in tiles.iterrows():
        tile_geom = tile_row.geometry
        tile_id = int(tile_row["tile_id"])

        ref_tile = subset_by_tile(ref_all, ref_all_sindex, tile_geom)
        cand_tile = subset_by_tile(cand_all, cand_all_sindex, tile_geom)

        if ref_tile.empty and cand_tile.empty:
            continue

        metrics, matches_df = compute_tile_metrics(
            ref_tile, cand_tile, TAU_OVERLAP, TAU_BUFFER_M, TAU_BOUNDARY_M, tile_id, ds_name
        )
        ds_tile_metrics.append(metrics)

        if not matches_df.empty:
            # Attach context info (city, dataset, tile_id)
            matches_df = matches_df.copy()
            matches_df["city"] = CITY
            matches_df["dataset"] = ds_name
            matches_df["tile_id"] = tile_id
            ds_match_rows.append(matches_df)

    # Save per-dataset tile metrics
    ds_tile_df = pd.DataFrame(ds_tile_metrics)
    tile_out_path = metrics_dir / f"vector_metrics_tiles_{ds_name}.parquet"
    ds_tile_df.to_parquet(tile_out_path, index=False)
    print(f"Saved tile metrics for {ds_name} → {tile_out_path}")

    # Save per-dataset matches
    if ds_match_rows:
        ds_matches_df = pd.concat(ds_match_rows, ignore_index=True)
        match_out_path = metrics_dir / f"vector_matches_{ds_name}.parquet"
        ds_matches_df.to_parquet(match_out_path, index=False)
        print(f"Saved building matches for {ds_name} → {match_out_path}")
    else:
        ds_matches_df = pd.DataFrame(columns=["ref_id", "cand_id", "iou", "area_ref", "area_cand", "rel_area_error", "city", "dataset", "tile_id"])

    all_tile_metrics.append(ds_tile_df)
    all_match_rows.append(ds_matches_df)

# Optional: combined city-wide dataframes
if all_tile_metrics:
    metrics_all = pd.concat(all_tile_metrics, ignore_index=True)
    metrics_all.to_parquet(metrics_dir / "vector_metrics_tiles_all_datasets.parquet", index=False)
    display(metrics_all.head())

if all_match_rows:
    matches_all = pd.concat(all_match_rows, ignore_index=True)
    matches_all.to_parquet(metrics_dir / "vector_matches_all_datasets.parquet", index=False)
    display(matches_all.head())

In [ ]:
# ---------------- Reporting (city-level summaries) ----------------
import pandas as pd

metrics_all_path = metrics_dir / "vector_metrics_tiles_all_datasets.parquet"
matches_all_path = metrics_dir / "vector_matches_all_datasets.parquet"

metrics_all = pd.read_parquet(metrics_all_path) if metrics_all_path.exists() else pd.DataFrame()
matches_all = pd.read_parquet(matches_all_path) if matches_all_path.exists() else pd.DataFrame()

def summarize_city(metrics_df: pd.DataFrame, matches_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for ds, mds in metrics_df.groupby("dataset"):
        # Totals from tiles
        tp = int(mds["tp"].sum())
        fp = int(mds["fp"].sum())
        fn = int(mds["fn"].sum())
        n_ref = int(mds["n_ref"].sum())
        n_cand = int(mds["n_cand"].sum())

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

        # Match-based distribution stats (TP-only)
        dsmatches = matches_df[matches_df["dataset"] == ds] if not matches_df.empty else pd.DataFrame()
        if not dsmatches.empty:
            ious = dsmatches["iou"].astype(float)
            bf = dsmatches["boundary_f_pair"].astype(float) if "boundary_f_pair" in dsmatches.columns else pd.Series(dtype=float)
            rel_area = dsmatches["rel_area_error"].astype(float)

            iou_mean = float(ious.mean())
            iou_median = float(ious.median())
            iou_p25 = float(ious.quantile(0.25))
            iou_p75 = float(ious.quantile(0.75))

            bf_mean = float(bf.mean()) if len(bf) else 0.0
            rel_area_mean = float(rel_area.mean())
            rel_area_median = float(rel_area.median())

            area_ref_sum = float(dsmatches["area_ref"].sum())
            area_cand_sum = float(dsmatches["area_cand"].sum())
            signed_area_bias = ((area_cand_sum - area_ref_sum) / area_ref_sum) if area_ref_sum > 0 else float("nan")
        else:
            iou_mean = iou_median = iou_p25 = iou_p75 = 0.0
            bf_mean = 0.0
            rel_area_mean = rel_area_median = float("nan")
            signed_area_bias = float("nan")

        rows.append({
            "city": CITY,
            "dataset": ds,
            "n_tiles": int(mds["tile_id"].nunique()),
            "n_ref_total": n_ref,
            "n_cand_total": n_cand,
            "tp_total": tp,
            "fp_total": fp,
            "fn_total": fn,
            "precision_city": precision,
            "recall_city": recall,
            "f1_city": f1,
            "iou_mean_tp": iou_mean,
            "iou_median_tp": iou_median,
            "iou_p25_tp": iou_p25,
            "iou_p75_tp": iou_p75,
            "boundary_f_meanpair_tp": bf_mean,
            "rel_area_error_mean_tp": rel_area_mean,
            "rel_area_error_median_tp": rel_area_median,
            "signed_area_bias_tp": signed_area_bias,
        })

    return pd.DataFrame(rows)

if not metrics_all.empty:
    city_summary = summarize_city(metrics_all, matches_all)
    summary_path = metrics_dir / "vector_city_summary_all_datasets.parquet"
    city_summary.to_parquet(summary_path, index=False)
    display(city_summary)

    # Also save CSV for quick sharing
    city_summary.to_csv(metrics_dir / "vector_city_summary_all_datasets.csv", index=False)
    print("Saved:", summary_path)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import datetime

metrics_all = pd.read_parquet(metrics_dir / "vector_metrics_tiles_all_datasets.parquet")

fig, ax = plt.subplots(figsize=(10, 4))  # optional but recommended for consistency
sns.boxplot(data=metrics_all, x="dataset", y="f1", ax=ax)

ax.set_title(f"Tile-level F1 scores – {CITY.capitalize()}")
ax.set_xlabel("Dataset")
ax.set_ylabel("F1")

save_figure(fig, fig_name("tile_f1_boxplot"))
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Load tiles and metrics
tiles_path = ROOT / f"data/02_interim/tiles/{CITY.lower()}_tiles.gpkg"
tiles = gpd.read_file(tiles_path)

metrics_all = pd.read_parquet(metrics_dir / "vector_metrics_tiles_all_datasets.parquet")

# Get the list of datasets present in the metrics table
datasets = metrics_all["dataset"].unique()
print("Datasets found:", datasets)

for ds_to_plot in datasets:
    # Filter metrics for this dataset
    metrics_ds = metrics_all[metrics_all["dataset"] == ds_to_plot]

    if metrics_ds.empty:
        print(f"Skipping {ds_to_plot}: no metrics available.")
        continue

    # Join F1 to tiles
    tiles_metrics = tiles.merge(
        metrics_ds[["tile_id", "f1"]],
        on="tile_id",
        how="left"
    )

    # Plot tile-level F1 for this dataset
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    tiles_metrics.plot(
        column="f1",
        ax=ax,
        legend=True,
        cmap="viridis",
        edgecolor="none"
    )
    ax.set_title(f"{CITY.capitalize()} – Tile-level F1 for {ds_to_plot}")
    ax.set_axis_off()

    save_figure(fig, fig_name(f"{CITY.capitalize()} – Tile-level F1 for {ds_to_plot}"))

    plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load metrics + matches
metrics_all = pd.read_parquet(metrics_dir / "vector_metrics_tiles_all_datasets.parquet")
matches_all = pd.read_parquet(metrics_dir / "vector_matches_all_datasets.parquet")

datasets = metrics_all["dataset"].unique()
print("Datasets found:", datasets, "\n")

for ds_to_plot in datasets:
    print(f"=== {CITY.capitalize()} — {ds_to_plot} ===")

    # --- Tile-level TP / FP / FN totals ---
    m_ds = metrics_all[metrics_all["dataset"] == ds_to_plot]

    tp = m_ds["tp"].sum()
    fp = m_ds["fp"].sum()
    fn = m_ds["fn"].sum()

    print(f"TP: {tp:,}  FP: {fp:,}  FN: {fn:,}")

    # --- IoU for TP matches only ---
    ious_tp = matches_all[matches_all["dataset"] == ds_to_plot]["iou"].dropna()

    # --- Add zeros for FP + FN to approximate 'all buildings' view ---
    num_zeros = fp + fn
    ious_all = pd.concat(
        [ious_tp, pd.Series(np.zeros(num_zeros))],
        ignore_index=True
    )

    print(f"TP-only IoU mean: {ious_tp.mean():.3f}  (n={len(ious_tp):,})")
    print(f"All buildings (TP + FP + FN→0) IoU mean: {ious_all.mean():.3f}  (n={len(ious_all):,})")

    # --- Plot histogram ---
    plt.figure(figsize=(6, 4))
    plt.hist(ious_all, bins=30)
    plt.title(f"{CITY.capitalize()} — IoU (TP + FP/FN as 0) — {ds_to_plot}")
    plt.xlabel("IoU")
    plt.ylabel("Count")
    plt.show()

    print()  # spacing between datasets


In [ ]:
# ---------------- USER SETTINGS ----------------
USE_EXPLICIT_BINS = True   # set to False to use quantile bins

# Explicit bins (used if USE_EXPLICIT_BINS=True)
size_bins = [0, 25, 50, 100, 500, 1000, np.inf] # 5 hist-equalize results in 25, 33, 46, 74, 8584 for Juba
size_bin_labels = [
    "<25",
    "25–50",
    "50–100",
    "100–500",
    "500–1000",
    ">1000"
]

# Quantile bins (used if USE_EXPLICIT_BINS=False)
n_bins = 5
# ------------------------------------------------

matches_all = pd.read_parquet(metrics_dir / "vector_matches_all_datasets.parquet")

datasets = matches_all["dataset"].unique()
print("Datasets found:", datasets)

for ds in datasets:
    print(f"\n=== {CITY.capitalize()} — {ds} ===")

    m_ds = matches_all[matches_all["dataset"] == ds].copy()

    if m_ds.empty:
        print("No matched buildings for this dataset.")
        continue

    # ---- Define building size bins ----
    if USE_EXPLICIT_BINS:
        m_ds["size_bin"] = pd.cut(
            m_ds["area_ref"],
            bins=size_bins,
            labels=size_bin_labels,
            include_lowest=True
        )
    else:
        m_ds["size_bin"] = pd.qcut(
            m_ds["area_ref"],
            q=n_bins,
            duplicates="drop"
        )

    # ---- Aggregate statistics per size bin ----
    size_stats = (
        m_ds
        .groupby("size_bin", observed=True)
        .agg(
            mean_iou=("iou", "mean"),
            median_iou=("iou", "median"),
            median_rel_area_error=("rel_area_error", "median"),
            count=("iou", "size")
        )
        .reset_index()
    )

    display(size_stats)

    # ---- Plot: IoU vs building size ----
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(
        size_stats["size_bin"].astype(str),
        size_stats["median_iou"],
        marker="o"
    )
    plt.xticks(rotation=45, ha="right")
    for label in ax.get_xticklabels():
      label.set_ha("right")
    ax.set_ylabel("Median IoU")
    ax.set_xlabel("Reference building size (m²)")
    ax.axhline(0, color="grey", linestyle="--", linewidth=1)
    ax.set_title(f"{CITY.capitalize()} – IoU vs building size – {ds}")
    ax.grid(True, axis="y", alpha=0.3)
    fig.tight_layout()

    save_figure(fig, fig_name(f"{CITY.capitalize()} – IoU vs building size – {ds}"))

    plt.show()
    plt.close(fig)

    # ---- Plot: Relative area error vs building size ----
    fig = plt.figure(figsize=(7, 4))
    plt.plot(
        size_stats["size_bin"].astype(str),
        size_stats["median_rel_area_error"],
        marker="o"
    )
    plt.xticks(rotation=45, ha="right")
    ax.set_ylabel("Median relative area error")
    ax.set_xlabel("Reference building size (m²)")
    ax.axhline(0, color="grey", linestyle="--", linewidth=1)
    ax.set_title(f"{CITY.capitalize()} – Relative area error vs building size – {ds}")
    ax.grid(True, axis="y", alpha=0.3)
    fig.tight_layout()

    save_figure(fig, fig_name(f"{CITY.capitalize()} – Relative area error vs building size – {ds}"))

    plt.show()